In [12]:
from langchain.tools import tool,InjectedToolArg
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated

Tool Creation

In [13]:
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """
    This function fetches the currency conversion factor between given base currency and target currency
    """
    url=f"https://v6.exchangerate-api.com/v6/9ec956aa00954b781f42dbea/pair/{base_currency}/{target_currency}"
    response=requests.get(url)
    return response.json()

@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """
    given a currency conversion rate this function calculates the target currency value from given base currency value
    """
    return base_currency_value*conversion_rate

In [3]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1783987201,
 'time_last_update_utc': 'Tue, 14 Jul 2026 00:00:01 +0000',
 'time_next_update_unix': 1784073601,
 'time_next_update_utc': 'Wed, 15 Jul 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.7212}

In [5]:
convert.invoke({'base_currency_value':10,'conversion_rate':95.7212})

957.212

Tool binding

In [18]:
llm=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation"
)
model=ChatHuggingFace(llm=llm,temperature=0.2)

llm_with_tools=model.bind_tools([get_conversion_factor,convert])

In [25]:
messages=[HumanMessage('What is the conversion factor of USD and INR, and based on that can you convert 10 usd to inr')]

In [26]:
ai_message=llm_with_tools.invoke(messages)

In [27]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_n0kofy2o8ca3eodrqfxmfjlk',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_slbrsjp50zdrs8a6p16ve9sg',
  'type': 'tool_call'}]

In [28]:
messages.append(ai_message)

In [29]:
import json
for tool_call in ai_message.tool_calls:
    # execute tool 1 & get conversion factor
    if tool_call['name']=='get_conversion_factor':
        tool_message1=get_conversion_factor.invoke(tool_call)
        conversion_rate=json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    # execute tool 2 for conversion
    if tool_call['name']== 'convert':
        tool_call['args']['conversion_rate']=conversion_rate
        tool_message2=convert.invoke(tool_call)
        messages.append(tool_message2)

In [30]:
messages

[HumanMessage(content='What is the conversion factor of USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_n0kofy2o8ca3eodrqfxmfjlk', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': 'call_slbrsjp50zdrs8a6p16ve9sg', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 342, 'total_tokens': 394}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f5f6b-14ba-70f3-a4b0-18e4157c8bd5-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_n0kofy2o8ca3eodrqfxmfjlk', 'typ

In [31]:
llm_with_tools.invoke(messages).content

'The conversion factor from USD to INR is approximately 95.7212. Based on this conversion rate, 10 USD would be approximately 957.212 INR.'